# 06. 北星論集（英文）掲載用の図

既存の分析結果（`03`の出力）から、英文原稿の図1・図2を作る。新しい推定は行わない。
- 図1：2023年の高所得クラブ40か国の散布図（log相対所得×log相対物価水準、回帰直線、日本・韓国・台湾を強調）
- 図2：年別横断面回帰の残差の推移（日本・韓国・台湾、2005〜2023年）

印刷はモノクロの可能性があるため、色に頼らず線種・マーカーで区別する。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
TABLES_DIR = BASE_DIR / "output" / "tables"
FIG_DIR = BASE_DIR / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

In [2]:
cs = pd.read_csv(PROCESSED_DIR / "penn_regression_2023_highincome.csv")
fig, ax = plt.subplots(figsize=(6.0, 4.2))
others = cs[~cs.countrycode.isin(["JPN", "KOR", "TWN"])]
ax.scatter(others.log_x, others.log_y, s=18, facecolors="white", edgecolors="0.35", linewidths=0.8, label="Other high-income economies")
xs = np.linspace(cs.log_x.min() - 0.05, cs.log_x.max() + 0.05, 50)
b = np.polyfit(cs.log_x, cs.log_y, 1)
ax.plot(xs, b[1] + b[0] * xs, color="0.2", lw=1.0, ls="--", label=f"OLS fit (slope {b[0]:.3f})")
for cc, name, mk in (("JPN", "Japan", "s"), ("KOR", "Korea", "^"), ("TWN", "Taiwan", "o")):
    r = cs[cs.countrycode == cc].iloc[0]
    ax.scatter(r.log_x, r.log_y, s=42, color="black", marker=mk, zorder=3)
    offs = {"JPN": (-8, 10), "KOR": (8, -4), "TWN": (8, 6)}[cc]
    ax.annotate(f"{name} ({r.resid_pct:+.1f}%)", (r.log_x, r.log_y), xytext=offs, textcoords="offset points",
                fontsize=9, ha="right" if cc == "JPN" else "left",
                bbox=dict(boxstyle="square,pad=0.1", fc="white", ec="none"))
ax.set_xlabel("log relative income (PPP GDP per capita, US = 1)")
ax.set_ylabel("log relative price level of consumption (US = 1)")
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_penn_2023_highincome.png", dpi=300)
print("slope", round(b[0], 3), "n", len(cs))

slope 1.013 n 40


In [3]:
ys = pd.read_csv(TABLES_DIR / "penn_regression_by_year_summary.csv")
fig, ax = plt.subplots(figsize=(6.0, 3.6))
for col, name, ls, mk in (("JPN_resid_pct", "Japan", "-", "s"), ("KOR_resid_pct", "Korea", "--", "^"),
                          ("TWN_resid_pct", "Taiwan", ":", "o")):
    ax.plot(ys.year, ys[col], color="black", ls=ls, marker=mk, ms=4, lw=1.0, label=name)
ax.axhline(0, color="0.5", lw=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Residual (% deviation from fitted price level)")
ax.set_xticks(range(2005, 2024, 2))
ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_residuals_by_year.png", dpi=300)
print(ys[["year", "slope", "n", "JPN_resid_pct", "KOR_resid_pct", "TWN_resid_pct"]].round(1).to_string(index=False))

 year  slope  n  JPN_resid_pct  KOR_resid_pct  TWN_resid_pct
 2005    0.6 26           11.5          -17.3          -54.0
 2006    0.7 28            5.7           -8.2          -54.1
 2007    0.7 28           -3.4          -12.5          -57.6
 2008    0.8 30            8.1          -23.8          -54.7
 2009    0.7 28           24.5          -31.1          -55.7
 2010    0.7 29           28.1          -24.6          -56.1
 2011    0.8 31           37.4          -19.6          -54.4
 2012    0.9 31           45.6          -12.4          -54.0
 2013    0.9 31           10.6           -4.4          -54.8
 2014    0.8 31            7.3           -2.2          -55.8
 2015    1.0 32           10.6            1.9          -51.9
 2016    1.1 33           34.5           -0.5          -50.1
 2017    1.0 36           29.8            1.6          -46.2
 2018    1.0 37           30.2           -2.1          -45.9
 2019    1.0 38           48.9            1.6          -44.0
 2020    1.0 35         

## 和文版（手元保管用）の図

同じデータ・同じ描き方で、ラベルだけ日本語にした図1・図2（`*_ja.png`）を作る。和文フォントは、使えるものを上から順に選ぶ。

In [4]:
from matplotlib import font_manager
JA_FONTS = ["Noto Serif JP", "Yu Mincho", "IPAexMincho", "MS Mincho"]
available = {f.name for f in font_manager.fontManager.ttflist}
ja_font = next(f for f in JA_FONTS if f in available)
ja_rc = {"font.family": ja_font, "axes.unicode_minus": False}

with plt.rc_context(ja_rc):
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    ax.scatter(others.log_x, others.log_y, s=18, facecolors="white", edgecolors="0.35", linewidths=0.8, label="その他の高所得国・地域")
    ax.plot(xs, b[1] + b[0] * xs, color="0.2", lw=1.0, ls="--", label=f"OLS回帰直線（傾き {b[0]:.3f}）")
    for cc, name, mk in (("JPN", "日本", "s"), ("KOR", "韓国", "^"), ("TWN", "台湾", "o")):
        r = cs[cs.countrycode == cc].iloc[0]
        ax.scatter(r.log_x, r.log_y, s=42, color="black", marker=mk, zorder=3)
        offs = {"JPN": (-8, 10), "KOR": (8, -4), "TWN": (8, 6)}[cc]
        ax.annotate(f"{name}（{r.resid_pct:+.1f}%）", (r.log_x, r.log_y), xytext=offs, textcoords="offset points",
                    fontsize=9, ha="right" if cc == "JPN" else "left",
                    bbox=dict(boxstyle="square,pad=0.1", fc="white", ec="none"))
    ax.set_xlabel("相対所得の対数（購買力平価ベース一人当たりGDP、米国＝1）")
    ax.set_ylabel("家計消費の相対物価水準の対数（米国＝1）")
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig1_penn_2023_highincome_ja.png", dpi=300)

    fig, ax = plt.subplots(figsize=(6.0, 3.6))
    for col, name, ls, mk in (("JPN_resid_pct", "日本", "-", "s"), ("KOR_resid_pct", "韓国", "--", "^"),
                              ("TWN_resid_pct", "台湾", ":", "o")):
        ax.plot(ys.year, ys[col], color="black", ls=ls, marker=mk, ms=4, lw=1.0, label=name)
    ax.axhline(0, color="0.5", lw=0.6)
    ax.set_xlabel("年")
    ax.set_ylabel("残差（予測物価水準からの乖離、%）")
    ax.set_xticks(range(2005, 2024, 2))
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig2_residuals_by_year_ja.png", dpi=300)
print("font:", ja_font)

findfont: Failed to find font weight normal, now using 200.


findfont: Failed to find font weight normal, now using 200.


findfont: Failed to find font weight normal, now using 200.


font: Noto Serif JP
